In [ ]:
"""
AI Research Paper Smart Intelligence System

"""

import os
import re
import pickle
import logging
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import numpy as np
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("arxiv_pipeline")


In [ ]:
# 0. CONFIG
from dataclasses import dataclass, field
from typing import List, Dict, Optional

@dataclass
class Config:
    dataset_name: str = "CShorten/ML-ArXiv-Papers"
    dataset_split: str = "train"

    # Embedding model. "all-MiniLM-L6-v2" is fast/general-purpose.
    # For research-paper-specific semantics, "allenai-specter" or
    # "sentence-transformers/allenai-specter" is a strong alternative.
    embedding_model_name: str = "all-MiniLM-L6-v2"
    embedding_batch_size: int = 128

    summarization_model_name: str = "facebook/bart-large-cnn"
    max_summary_input_words: int = 800   # BART has a token limit; abstracts are short, this is headroom

    keybert_top_n: int = 8

    spacy_model_name: str = "en_core_web_sm"

    faiss_index_path: str = "arxiv_faiss.index"
    metadata_path: str = "arxiv_metadata.pkl"

    device: str = "cuda"  # falls back to cpu automatically if unavailable


CONFIG = Config()

In [ ]:
# DATA EXTRACTION
def load_papers(config: Config, sample_size: Optional[int] = None) -> List[Dict]:

    import pandas as pd
    from huggingface_hub import hf_hub_download

    logger.info(f"Downloading CSV for dataset '{config.dataset_name}' ...")
    csv_path = hf_hub_download(
        repo_id=config.dataset_name,
        filename="ML-Arxiv-Papers.csv",
        repo_type="dataset",
    )
    df = pd.read_csv(csv_path)

    if sample_size is not None:
        df = df.head(sample_size)

    ds = df.to_dict(orient="records")
    logger.info(f"Loaded {len(ds)} records. Extracting title/abstract ...")

    # The dataset's known columns are 'title' and 'abstract'; guard against
    # naming variants just in case the schema differs across versions.
    column_names = list(df.columns)
    cols = {c.lower(): c for c in column_names}
    title_col = cols.get("title")
    abstract_col = cols.get("abstract")
    if title_col is None or abstract_col is None:
        raise ValueError(f"Expected 'title'/'abstract' columns, got: {column_names}")

    papers = []
    for i, row in enumerate(tqdm(ds, desc="Extracting")):
        title = str(row.get(title_col) or "")
        abstract = str(row.get(abstract_col) or "")
        if not title.strip() or not abstract.strip():
            continue
        papers.append({"id": i, "title": title, "abstract": abstract})

    logger.info(f"Extracted {len(papers)} valid (title, abstract) pairs.")
    return papers

In [ ]:
# CLEANING
_WHITESPACE_RE = re.compile(r"\s+")
_LATEX_RE = re.compile(r"\$.*?\$|\\[a-zA-Z]+\{.*?\}")
_URL_RE = re.compile(r"http\S+|www\.\S+")
_NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9.,;:%\-\s]")


def clean_text(text: str) -> str:
    """
    Cleans raw paper text: strips LaTeX artifacts, URLs, boilerplate
    line breaks, and stray symbols, while preserving natural sentence
    structure (important for embeddings & summarization quality).
    """
    text = text.replace("\n", " ")
    text = _LATEX_RE.sub(" ", text)
    text = _URL_RE.sub(" ", text)
    text = _NON_ALNUM_RE.sub(" ", text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text


def clean_papers(papers: List[Dict]) -> List[Dict]:
    logger.info("Cleaning title/abstract text ...")
    for p in tqdm(papers, desc="Cleaning"):
        p["title_clean"] = clean_text(p["title"])
        p["abstract_clean"] = clean_text(p["abstract"])
        # This is what gets embedded / summarized: natural, cleaned prose.
        p["full_text"] = f"{p['title_clean']}. {p['abstract_clean']}"
    return papers


In [ ]:
# PREPROCESSING: TOKENIZATION + STOPWORD REMOVAL
def ensure_nltk_resources():
    import nltk
    for resource in ["punkt", "punkt_tab", "stopwords"]:
        try:
            nltk.data.find(f"tokenizers/{resource}" if "punkt" in resource else f"corpora/{resource}")
        except LookupError:
            nltk.download(resource, quiet=True)


def preprocess_papers(papers: List[Dict]) -> List[Dict]:

    ensure_nltk_resources()
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords

    stop_words = set(stopwords.words("english"))

    logger.info("Tokenizing + removing stopwords ...")
    for p in tqdm(papers, desc="Preprocessing"):
        tokens = word_tokenize(p["full_text"].lower())
        tokens_clean = [t for t in tokens if t.isalpha() and t not in stop_words]
        p["tokens"] = tokens
        p["tokens_no_stopwords"] = tokens_clean
    return papers



In [ ]:
import transformers
print(transformers.__version__)
print(transformers.__file__)
from transformers.pipelines import SUPPORTED_TASKS
print('summarization' in SUPPORTED_TASKS)

5.15.0
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py
False


In [ ]:
!pip uninstall -y transformers
!pip install "transformers<5.0" --no-cache-dir

Found existing installation: transformers 5.15.0
Uninstalling transformers-5.15.0:
  Successfully uninstalled transformers-5.15.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 119.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.27.0
    Uninstalling huggingface_hub-1.27.0:
      Successfully uninstalled huggingface_hub-1.27.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
!pip install "transformers==4.46.3" --no-cache-dir --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 106.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 135.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 141.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 152.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 166.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 169.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import transformers; print(transformers.__version__)

4.46.3


In [ ]:
!pip install "transformers==4.46.3" sentencepiece accelerate --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 201.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 182.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of t

In [ ]:
import transformers
print(transformers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.46.3


In [ ]:
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)
print(summarizer("This is a test sentence to confirm the summarization pipeline loads and runs correctly." * 5))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Your max_length is set to 142, but your input_length is only 82. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=41)


[{'summary_text': 'This is a test sentence to confirm the summarization pipeline loads and runs correctly. This is a Test Sentence to confirm that a function has been applied to a function. This test sentence confirms that the function was applied to the function that was supposed to be applied to. The function was intended to be used to test whether the function had been applied properly.'}]


In [ ]:
!pip install datasets sentence-transformers faiss-cpu "transformers<5.0" keybert spacy nltk torch --upgrade --no-cache-dir
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 139.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 200.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 172.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 186.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 155.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.7/32.7 MB 247.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 235.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# CONTEXTUAL EMBEDDINGS + FAISS INDEX
class EmbeddingIndex:
    def __init__(self, config: Config):
        self.config = config
        self.model = None
        self.index = None
        self.dim = None

    def _load_model(self):
        if self.model is None:
            from sentence_transformers import SentenceTransformer
            import torch
            device = self.config.device if torch.cuda.is_available() else "cpu"
            logger.info(f"Loading Sentence-Transformer '{self.config.embedding_model_name}' on {device} ...")
            self.model = SentenceTransformer(self.config.embedding_model_name, device=device)
        return self.model

    def build(self, papers: List[Dict]):
        import faiss

        model = self._load_model()
        texts = [p["full_text"] for p in papers]

        logger.info("Encoding papers into contextual embeddings ...")
        embeddings = model.encode(
            texts,
            batch_size=self.config.embedding_batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,  # so inner product == cosine similarity
        ).astype("float32")

        self.dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(self.dim)
        self.index.add(embeddings)
        logger.info(f"FAISS index built with {self.index.ntotal} vectors (dim={self.dim}).")
        return embeddings

    def search(self, query: str, k: int = 5) -> List[Dict]:
        model = self._load_model()
        q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        scores, indices = self.index.search(q_emb, k)
        return [{"idx": int(i), "score": float(s)} for i, s in zip(indices[0], scores[0]) if i != -1]

    def save(self):
        import faiss
        faiss.write_index(self.index, self.config.faiss_index_path)
        logger.info(f"Saved FAISS index -> {self.config.faiss_index_path}")

    def load(self):
        import faiss
        self.index = faiss.read_index(self.config.faiss_index_path)
        logger.info(f"Loaded FAISS index <- {self.config.faiss_index_path}")

In [ ]:
# SUMMARIZATION (BART)
class Summarizer:
    def __init__(self, config: Config):
        self.config = config
        self._pipe = None

    def _load(self):
        if self._pipe is None:
            from transformers import pipeline
            import torch
            device = 0 if torch.cuda.is_available() else -1
            logger.info(f"Loading BART summarizer '{self.config.summarization_model_name}' ...")
            self._pipe = pipeline("summarization", model=self.config.summarization_model_name, device=device)
        return self._pipe

    def summarize(self, text: str, max_length: int = 60, min_length: int = 15) -> str:
        pipe = self._load()
        words = text.split()
        if len(words) > self.config.max_summary_input_words:
            text = " ".join(words[: self.config.max_summary_input_words])
        result = pipe(text, max_length=max_length, min_length=min_length, do_sample=False)
        return result[0]["summary_text"]


In [ ]:
!pip install keybert --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 46.2 MB/s eta 0:00:00


In [ ]:
!pip install torch torchvision torchaudio --no-cache-dir


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.5 MB/s eta 0:00:00


In [ ]:
import torch, torchvision
print(torch.__version__)
print(torchvision.__version__)
from torchvision.ops import nms
print('nms OK')

2.13.0+cu130
0.28.0+cu130
nms OK


In [ ]:
# KEYWORD EXTRACTION (KeyBERT)
class KeywordExtractor:
    def __init__(self, config: Config, sbert_model=None):
        self.config = config
        self._kw_model = None
        self._sbert_model = sbert_model  # reuse the same Sentence-Transformer if available

    def _load(self):
        if self._kw_model is None:
            from keybert import KeyBERT
            logger.info("Loading KeyBERT ...")
            self._kw_model = KeyBERT(model=self._sbert_model or self.config.embedding_model_name)
        return self._kw_model

    def extract(self, text: str, top_n: Optional[int] = None) -> List[str]:
        kw_model = self._load()
        top_n = top_n or self.config.keybert_top_n
        pairs = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            use_mmr=True,
            diversity=0.5,
            top_n=top_n,
        )
        return [kw for kw, _score in pairs]


In [ ]:
# NER: general entities + custom ML tech-stack recognition
# Generic spaCy models (en_core_web_sm) are trained on newswire text and do
# NOT know that "PyTorch" is a framework or "Python" is a language in this
# ML-paper sense. So we combine:
#   (a) spaCy's built-in NER for general entities (ORG, PERSON, GPE, etc.)
#   (b) a curated PhraseMatcher over ML-specific vocab, tagged with the
#       category it belongs to (LANGUAGE, FRAMEWORK, LIBRARY, ALGORITHM, DATASET)
# This hybrid is the standard practical approach when no off-the-shelf model
# covers a specialized technical domain.

ML_TECH_VOCAB: Dict[str, List[str]] = {
    "LANGUAGE": ["Python", "C++", "Java", "R", "Julia", "JavaScript", "Rust", "Scala"],
    "FRAMEWORK": [
        "PyTorch", "TensorFlow", "Keras", "JAX", "MXNet", "Caffe", "Theano",
        "ONNX", "Hugging Face Transformers", "FastAI", "PaddlePaddle",
    ],
    "LIBRARY": [
        "scikit-learn", "NumPy", "Pandas", "SciPy", "OpenCV", "NLTK", "spaCy",
        "Matplotlib", "XGBoost", "LightGBM", "FAISS", "Gensim", "sentence-transformers",
    ],
    "ALGORITHM": [
        "transformer", "BERT", "GPT", "LSTM", "CNN", "RNN", "GAN", "attention mechanism",
        "reinforcement learning", "random forest", "gradient boosting", "autoencoder",
        "diffusion model", "graph neural network",
    ],
    "DATASET": ["ImageNet", "MNIST", "CIFAR-10", "COCO", "SQuAD", "GLUE", "WikiText"],
}


class TechStackNER:
    def __init__(self, config: Config):
        self.config = config
        self._nlp = None

    def _load(self):
        if self._nlp is None:
            import spacy
            from spacy.matcher import PhraseMatcher

            logger.info(f"Loading spaCy model '{self.config.spacy_model_name}' ...")
            try:
                self._nlp = spacy.load(self.config.spacy_model_name)
            except OSError:
                raise OSError(
                    f"spaCy model '{self.config.spacy_model_name}' not found. "
                    f"Run: python -m spacy download {self.config.spacy_model_name}"
                )

            matcher = PhraseMatcher(self._nlp.vocab, attr="LOWER")
            for label, terms in ML_TECH_VOCAB.items():
                patterns = [self._nlp.make_doc(t) for t in terms]
                matcher.add(label, patterns)
            self._matcher = matcher
        return self._nlp

    def extract(self, text: str) -> Dict[str, List[str]]:
        nlp = self._load()
        doc = nlp(text)

        results: Dict[str, List[str]] = {"GENERAL": []}
        for ent in doc.ents:
            results["GENERAL"].append(f"{ent.text} ({ent.label_})")

        for match_id, start, end in self._matcher(doc):
            label = nlp.vocab.strings[match_id]
            span_text = doc[start:end].text
            results.setdefault(label, [])
            if span_text not in results[label]:
                results[label].append(span_text)

        return results


In [ ]:
# ORCHESTRATOR
class ArxivPaperIntelligenceSystem:
    def __init__(self, config: Config = CONFIG):
        self.config = config
        self.papers: List[Dict] = []
        self.embedder = EmbeddingIndex(config)
        self.summarizer = Summarizer(config)
        self.keyword_extractor: Optional[KeywordExtractor] = None
        self.ner = TechStackNER(config)

    def build(self, sample_size: Optional[int] = None, save: bool = True):
        """Runs steps 1-5: extract -> clean -> preprocess -> embed -> index."""
        self.papers = load_papers(self.config, sample_size=sample_size)
        self.papers = clean_papers(self.papers)
        self.papers = preprocess_papers(self.papers)
        self.embedder.build(self.papers)
        self.keyword_extractor = KeywordExtractor(self.config, sbert_model=self.embedder.model)

        if save:
            self.embedder.save()
            with open(self.config.metadata_path, "wb") as f:
                pickle.dump(self.papers, f)
            logger.info(f"Saved paper metadata -> {self.config.metadata_path}")

    def load(self):
        """Loads a previously built index + metadata (skip re-embedding 100k+ papers)."""
        with open(self.config.metadata_path, "rb") as f:
            self.papers = pickle.load(f)
        self.embedder._load_model()
        self.embedder.load()
        self.keyword_extractor = KeywordExtractor(self.config, sbert_model=self.embedder.model)

    def search(self, query: str, k: int = 5) -> List[Dict]:
        """Step 6: semantic search -> top-k most relevant papers."""
        hits = self.embedder.search(query, k=k)
        results = []
        for h in hits:
            paper = self.papers[h["idx"]]
            results.append({**paper, "similarity_score": h["score"]})
        return results

    def analyze(self, paper: Dict) -> Dict:
        """Steps 7-9 for a single retrieved paper: summary, keywords, tech-stack NER."""
        summary = self.summarizer.summarize(paper["full_text"])
        keywords = self.keyword_extractor.extract(paper["full_text"])
        entities = self.ner.extract(paper["full_text"])
        return {"summary": summary, "keywords": keywords, "entities": entities}

    def query(self, question: str, k: int = 5, analyze_top: int = 3) -> List[Dict]:
        """
        Full end-to-end call: semantic search + summarize/keyword/NER-analyze
        the top `analyze_top` hits (analysis is the expensive part, so it's
        only run on the most relevant results, not the full k).
        """
        results = self.search(question, k=k)
        for i, r in enumerate(results):
            if i < analyze_top:
                r.update(self.analyze(r))
        return results

    @staticmethod
    def explain(results: List[Dict]):
        for rank, r in enumerate(results, start=1):
            print(f"\n[{rank}] {r['title_clean']}  (score={r['similarity_score']:.3f})")
            if "summary" in r:
                print(f"    Summary : {r['summary']}")
            if "keywords" in r:
                print(f"    Keywords: {', '.join(r['keywords'])}")
            if "entities" in r:
                tech = {k: v for k, v in r["entities"].items() if k != "GENERAL" and v}
                if tech:
                    print(f"    Tech stack: {tech}")


In [ ]:
# example

import pandas as pd
from huggingface_hub import hf_hub_download

# Install faiss-cpu to ensure it's available
!pip install faiss-cpu

# 1. Load a small sample of papers
csv_path = hf_hub_download(repo_id="CShorten/ML-ArXiv-Papers", filename="ML-Arxiv-Papers.csv", repo_type="dataset")
df = pd.read_csv(csv_path).head(200)   # small sample for a quick test
papers = [f"{t}. {a}" for t, a in zip(df["title"], df["abstract"])]
print(f"Loaded {len(papers)} papers.")

# 2. Embed with Sentence-Transformers + build FAISS index
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(papers, show_progress_bar=True, normalize_embeddings=True).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} papers.")

# 3. Semantic search
query = "transformer-based models for time series forecasting"
q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
scores, idxs = index.search(q_emb, k=3)

top_papers = [papers[i] for i in idxs[0]]
print("\nTop matches:")
for rank, (i, s) in enumerate(zip(idxs[0], scores[0]), start=1):
    print(f"[{rank}] score={s:.3f} -> {df.iloc[i]['title']}")

# 4. Summarize the top result with BART
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)
summary = summarizer(top_papers[0][:2000], max_length=60, min_length=15, do_sample=False)[0]["summary_text"]
print(f"\nSummary of top match:\n{summary}")

# 5. Keywords with KeyBERT
from keybert import KeyBERT
kw_model = KeyBERT(model=embedder)
keywords = kw_model.extract_keywords(top_papers[0], keyphrase_ngram_range=(1, 2), stop_words="english", top_n=8)
print(f"\nKeywords: {[k for k, _ in keywords]}")

# 6. NER for tech-stack terms (simple keyword match, no spaCy setup needed)
ML_TECH_TERMS = ["Python", "PyTorch", "TensorFlow", "Keras", "scikit-learn", "BERT", "GPT", "LSTM", "CNN", "transformer"]
found = [t for t in ML_TECH_TERMS if t.lower() in top_papers[0].lower()]
print(f"\nTech-stack terms found: {found}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 41.8 MB/s eta 0:00:00
Loaded 200 papers.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

FAISS index built with 200 papers.

Top matches:
[1] score=0.392 -> Defensive forecasting for optimal prediction with expert advice
[2] score=0.342 -> On Sequences with Non-Learnable Subsequences
[3] score=0.332 -> A game-theoretic version of Oakes' example for randomized forecasting

Summary of top match:
The method of defensive forecasting is applied to the problem of prediction with expert advice. It turns out that defensive forecastingis not only competitive with the Aggregating Algorithm but also handles the case of "second-guessing" experts.

Keywords: ['defensive forecasting', 'prediction expert', 'learner prediction', 'optimal prediction', 'forecasting optimal', 'forecasting', 'prediction', 'forecasting competitive']

Tech-stack terms found: []
